In [ ]:
! pip install datasets accelerate evaluate peft
! pip install git+https://github.com/huggingface/transformers
! pip install sacrebleu
! pip install bert-score
! pip install evaluate rouge-score
! pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-f0ivugky
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-f0ivugky
  Resolved https://github.com/huggingface/transformers to commit 52cb0653b48fcb0737a74546911df77034b61732
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11359424 sha256=32574f99f184b2484ef9208b5317d93911558411de673e9e418151d041e6acef
  Stored in directory: /tmp/pip-ephem-wheel-cache-vudelz9m/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfu

In [ ]:
import torch
from datasets import load_from_disk

from google.colab import drive
drive.mount('/content/gdrive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Mounted at /content/gdrive
Using device: cuda


## Load In Dataset

In [ ]:
ds = load_from_disk("/content/gdrive/MyDrive/musiccaps_with_audio")
print(len(ds))

print(ds[0].keys())

4204
dict_keys(['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval', 'audio', 'download_status'])


## Split into Training and Testing sets

In [ ]:
# ds = ds.select(range(200))

splits1 = ds.train_test_split(test_size=0.3, seed=42)  # 70% train, 30% temp
train_ds = splits1['train']
temp_ds = splits1['test']

splits2 = temp_ds.train_test_split(test_size=0.5, seed=42)  # 50% val, 50% test

val_ds = splits2['train']
test_ds = splits2['test']

print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

2942
631
631


## Load in Model and Processor

In [ ]:
from io import BytesIO
from urllib.request import urlopen
import librosa
from transformers import BitsAndBytesConfig, AutoProcessor, Qwen2AudioForConditionalGeneration

In [ ]:
# For 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Or "fp4"
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)

model = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B", quantization_config=quantization_config,
                                                           device_map="auto", trust_remote_code=True)
model.to(device)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B" ,trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Process the Dataset for training

In [ ]:
import numpy as np
import copy
def process_data(examples):
    # prompt for text generation
    prompt = "<|audio_bos|><|AUDIO|><|audio_eos|>Describe the music thoroughly:"

    full_texts = [
        f"<|audio_bos|><|AUDIO|><|audio_eos|>Describe the music thoroughly: {caption}"
        for caption in examples["caption"]
    ]

    # resampling audio from sr 44100 -> 16000 for processor
    audio_arrays = []
    for a in examples["audio"]:
        waveform = a["array"]
        sr = a["sampling_rate"]
        waveform_16k = librosa.resample(waveform, orig_sr=sr, target_sr=16000)
        audio_arrays.append(waveform_16k)

    #processes audio and text prompt
    inputs = processor(text=full_texts, audio=audio_arrays, sampling_rate=16000, padding=False, return_tensors="np")

    prompt_ids = processor.tokenizer(prompt).input_ids
    # print(prompt_ids)

    # masks complete prompt from labels to compute loss/metrics only on actual caption
    labels = copy.deepcopy(inputs["input_ids"])
    label_ids = processor.tokenizer(examples["caption"]).input_ids
    for i in range(len(label_ids)):
        prompt_range = len(labels[i]) - len(label_ids[i])
        labels[i][:prompt_range] = -100

    inputs["labels"] = labels

    # print(inputs["input_ids"].shape)
    # print(inputs["attention_mask"].shape)
    # print(inputs["input_features"].shape)
    # print(inputs["labels"].shape)

    return inputs

In [ ]:
# example for one sample
sample = train_ds[0:2]
processed = process_data(sample)
for key, value in processed.items():
    print(key)
    # print the first item if value is a list, tensor, or something indexable
    try:
        print("First item:", value[0])
    except (TypeError, IndexError):
        # fallback if not indexable
        print("Value (not indexable):", value)
    print("-" * 30)  # separator for readability
#print(processed.keys())

input_ids
First item: [151647 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 151646 151646 151646
 151646 151646 151646 151646 151646 151646 151646 15164

In [ ]:
for k, v in processed.items():
    print(k, type(v), len(v))

input_ids <class 'numpy.ndarray'> 2
attention_mask <class 'numpy.ndarray'> 2
input_features <class 'numpy.ndarray'> 2
feature_attention_mask <class 'numpy.ndarray'> 2
labels <class 'numpy.ndarray'> 2


In [ ]:
import librosa
import requests
from io import BytesIO

urls = [
    "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3",
    "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-2.mp3"
]

audio_arrays = []
sr_model = processor.feature_extractor.sampling_rate  # usually 16000

for url in urls:
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    audio, sr = librosa.load(BytesIO(r.content), sr=sr_model)  # resampled automatically
    audio_arrays.append(audio)

prompt = "<|audio_bos|><|AUDIO|><|audio_eos|>Describe the music thoroughly:"
prompts = [prompt, prompt]  # same text for both audios

inputs = processor(
    text=prompts,
    audio=audio_arrays,
    return_tensors="pt",
    padding=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

generated_ids = model.generate(**inputs, max_new_tokens=128)

# remove prompt tokens from generation
generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

captions = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

for i, cap in enumerate(captions):
    print(f"Audio {i+1} caption: {cap}")

It is strongly recommended to pass the `sampling_rate` argument to `WhisperFeatureExtractor()`. Failing to do so can result in silent errors that might be hard to debug.


Audio 1 caption:  "The Perfect Day" is a track by the artist 'The Perfect Day', featured in their album "The Perfect Day". This song was released on YouTube by CDBaby and auto-generated by YouTube. It was provided to YouTube by TuneCore and was released under Perfect Day Records in 2016.
Audio 2 caption:  This techno track has a high-energy, driving rhythm that is perfect for dancing. The use of synthesizers and drum machines creates a futuristic sound that is both captivating and exciting.


In [ ]:
del inputs
del generated_ids
del captions

In [ ]:
proc_train_ds = train_ds.map(process_data, batched=True,
                             remove_columns=train_ds.column_names) # removes unneeded columns
proc_val_ds = val_ds.map(process_data, batched=True,
                           remove_columns=test_ds.column_names) # removes unneeded columns

# Ensure all relevant columns are included in the PyTorch format
proc_train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "input_features", "feature_attention_mask", "labels"])
proc_val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "input_features", "feature_attention_mask", "labels"])

Map:   0%|          | 0/631 [00:00<?, ? examples/s]

In [ ]:
print("Processed training dataset columns:")

for col in proc_train_ds.column_names:
    data = proc_train_ds[col]  # get the full column
    print(f"Column: {col}")
    print(f"  Type of first item: {type(data[0])}")
    print(f"  Length of column: {len(data)}")
    print("-" * 30)

print("Test dataset columns:")
for col in proc_val_ds.column_names:
    data = proc_val_ds[col]  # get the full column
    print(f"Column: {col}")
    print(f"  Type of first item: {type(data[0])}")
    print(f"  Length of column: {len(data)}")
    print("-" * 30)

Processed training dataset columns:
Column: input_ids
  Type of first item: <class 'torch.Tensor'>
  Length of column: 2942
------------------------------
Column: attention_mask
  Type of first item: <class 'torch.Tensor'>
  Length of column: 2942
------------------------------
Column: input_features
  Type of first item: <class 'torch.Tensor'>
  Length of column: 2942
------------------------------
Column: feature_attention_mask
  Type of first item: <class 'torch.Tensor'>
  Length of column: 2942
------------------------------
Column: labels
  Type of first item: <class 'torch.Tensor'>
  Length of column: 2942
------------------------------
Test dataset columns:
Column: input_ids
  Type of first item: <class 'torch.Tensor'>
  Length of column: 631
------------------------------
Column: attention_mask
  Type of first item: <class 'torch.Tensor'>
  Length of column: 631
------------------------------
Column: input_features
  Type of first item: <class 'torch.Tensor'>
  Length of column

## Make Evaluation function (BERT, BLEU, METEOR, etc)

In [ ]:
def generate_test_responses(examples):
  prompts = [f"<|audio_bos|><|AUDIO|><|audio_eos|>Describe the music thoroughly:" for _ in examples["audio"]]
  # resampling audio from sr 44100 -> 16000 for processor
  audio_arrays = []
  for a in examples["audio"]:
    waveform = a["array"]
    sr = a["sampling_rate"]
    waveform_16k = librosa.resample(waveform, orig_sr=sr, target_sr=16000)
    audio_arrays.append(waveform_16k)

  #processes audio and text prompt
  inputs = processor(text=prompts, audio=audio_arrays, sampling_rate=16000, padding=True, return_tensors="pt")
  captions = processor.tokenizer(examples["caption"], padding=True).input_ids

  inputs = {k: v.to(model.device) for k, v in inputs.items()}
  generated_ids = model.generate(**inputs, max_new_tokens=128)

  # remove prompt tokens from generation
  generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

  return {"predictions":generated_ids, "references": captions}

In [ ]:
import nltk
import sacrebleu
import numpy as np
from bert_score import score
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
import evaluate
import torch # Added import

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

from bert_score import BERTScorer

# Load once at the start
scorer = BERTScorer(model_type="roberta-base", lang="en", rescale_with_baseline=True)

rouge = evaluate.load('rouge')

def compute_metrics(eval_set):
    # eval_set["predictions"] is a list of torch.Tensor (one per example in the dataset column)
    # eval_set["references"] is a list of lists of ints (one per example in the dataset column)

    predictions = [torch.tensor(p, device='cpu') if not isinstance(p, torch.Tensor) else p.to('cpu')
                   for p in eval_set["predictions"]]
    references = [torch.tensor(r, device='cpu') if not isinstance(r, torch.Tensor) else r.to('cpu')
                  for r in eval_set["references"]]

    # print(f"Pred shape: {len(predictions)}x")
    print("First pred IDs:", predictions[0][:20])

    # Decode predictions and references
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = processor.batch_decode(references, skip_special_tokens=True)

    print("\n=== SAMPLE PREDICTIONS ===")
    for i in range(min(3, len(decoded_preds))): # Ensure we don't go out of bounds
        print("PRED:", decoded_preds[i])
        print("REF :", decoded_refs[i])
        print()
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_refs = [r.strip() for r in decoded_refs]

    # BLEU Score
    bleu_score = sacrebleu.corpus_bleu(decoded_preds, [decoded_refs])

    # METEOR
    meteor_scores = []
    for pred, refs in zip(decoded_preds, decoded_refs):
        pred_tokens = pred.split()
        ref_tokens = refs.split()
        m_score = meteor_score([ref_tokens], pred_tokens)
        meteor_scores.append(m_score)
    meteor = sum(meteor_scores)/len(meteor_scores) if meteor_scores else 0.0 # Handle empty list

    # ROUGE
    rouge_score = rouge.compute(predictions=decoded_preds, references=decoded_refs)

    # BERT-score: Filter out empty strings and ensure aligned pairs
    aligned_preds = []
    aligned_refs = []
    for pred, ref in zip(decoded_preds, decoded_refs):
        if pred and ref: # Only include pairs where both are non-empty
            aligned_preds.append(pred)
            aligned_refs.append(ref)

    bert_precision, bert_recall, bert_f1 = 0.0, 0.0, 0.0
    if aligned_preds and aligned_refs:
        bert_precision, bert_recall, bert_f1 = scorer.score(aligned_preds, aligned_refs, batch_size=16) # Reduced batch size
        bert_precision = bert_precision.mean().item()
        bert_recall = bert_recall.mean().item()
        bert_f1 = bert_f1.mean().item()

    return {
        "BLEU": bleu_score.score,
        "METEOR": meteor,
        "ROUGE-1": rouge_score["rouge1"],
        "ROUGE-2": rouge_score["rouge2"],
        "ROUGE-L": rouge_score["rougeL"],
        "BERT_Precision": bert_precision,
        "BERT_Recall": bert_recall,
        "BERT_F1": bert_f1
    }

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Test scores on validation set before training

In [ ]:
val_set_metrics = val_ds.map(generate_test_responses, batched=True, batch_size=100,
                             remove_columns=test_ds.column_names) # removes unneeded columns
val_set_metrics.set_format(type="torch", columns=["predictions", "references"])
eval_results = compute_metrics(val_set_metrics)

for key, value in eval_results.items():
        metric_name = key.title()
        print(f"{metric_name:.<30} {value:.4f}")

Map:   0%|          | 0/631 [00:00<?, ? examples/s]

First pred IDs: tensor([ 1096,   374,   264,  3887, 14633,   315,   264, 26096,  8968, 13768,
         7055,  5619,    13,   576,  8968, 86720,   525,  5619,   279, 61584])

=== SAMPLE PREDICTIONS ===
PRED:  This is a live recording of a Scottish bagpipe band playing. The bagpipes are playing the melody while the snare drum, bass drum and cymbals are providing the rhythm in the background. There are no other instruments in this piece. The mood of this piece is lively. This piece could be played as part of the background music for a drama movie where people are playing bagpipes.
REF : This is the recording of a Pakistani marching band. The bagpipes are playing the melody. The rhythmic background is provided by a drumline that is played by the snare drums, the bass drums and the cymbals. The atmosphere is vibrant and lively. This piece could be playing in the background at a parade.

PRED:  This song features a blues guitar playing a clean tone. The male voice describes what he is doing 

In [ ]:
del val_set_metrics
del eval_results

## Set up training and fine tune the model

In [ ]:
# train with LoRA
from peft import LoraConfig, get_peft_model
config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 5,505,024 || all params: 8,402,599,936 || trainable%: 0.0655


In [ ]:
print(processor.tokenizer.padding_side)

left


In [ ]:
# collator to pad each batch the same amount
def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    input_features = [item["input_features"] for item in batch]
    feature_masks = [item["feature_attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    # ---- TEXT PADDING ----
    padded = processor.tokenizer.pad(
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        },
        return_tensors="pt"
    )

    # ---- LABEL PADDING ----
    labels = torch.nn.utils.rnn.pad_sequence(
        [l for l in labels],
        batch_first=True,
        padding_value=-100,
        padding_side='left'
    )

    # ---- AUDIO PADDING ----
    max_audio_len = max(f.shape[-1] for f in input_features)

    padded_features = []
    padded_feature_masks = []

    for f, m in zip(input_features, feature_masks):
        pad_size = max_audio_len - f.shape[-1]

        if pad_size > 0:
            f = torch.nn.functional.pad(f, (0, pad_size))
            m = torch.nn.functional.pad(m, (0, pad_size))  # pad mask too

        padded_features.append(f)
        padded_feature_masks.append(m)

    input_features = torch.stack(padded_features)
    feature_attention_mask = torch.stack(padded_feature_masks)

    return {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
        "input_features": input_features,
        "feature_attention_mask": feature_attention_mask,
        "labels": labels,
    }

In [ ]:
# check collate_fn
batch = collate_fn([proc_train_ds[0], proc_train_ds[1]])

print(batch["input_features"].shape)
print(batch["labels"].shape)
print((batch["labels"] == -100).sum())

for key, value in batch.items():
    print(key)
    # print the first item if value is a list, tensor, or something indexable
    try:
        print("First item:", value[1])
    except (TypeError, IndexError):
        # fallback if not indexable
        print("Value (not indexable):", value)
    print("-" * 30)  # separator for readability


torch.Size([2, 128, 3000])
torch.Size([2, 319])
tensor(535)
input_ids
First item: tensor([151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151647, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646, 151646,
        151646, 151646, 151646, 151646

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

# Ensure use_cache is False when gradient_checkpointing is True
# model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="./audio-qwen-lora",
    eval_strategy='epoch',
    logging_strategy="epoch",
    learning_rate = 1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    report_to="none",
    fp16=True,
    # gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    # gradient_checkpointing_kwargs = {'use_reentrant': False}
)

# just with model to freeze processor
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=proc_train_ds,
    eval_dataset=proc_val_ds,
    # compute_metrics=compute_metrics,
    data_collator=collate_fn
)

In [ ]:
train_results = trainer.train()

Epoch,Training Loss,Validation Loss
1,10.689168,1.245216
2,9.425976,1.213018
3,8.839424,1.206094


## Metrics on Val set After Training

In [ ]:
val_set_metrics = val_ds.map(generate_test_responses, batched=True, batch_size=80,
                             remove_columns=test_ds.column_names) # removes unneeded columns
eval_results = compute_metrics(val_set_metrics)

for key, value in eval_results.items():
        metric_name = key.title()
        print(f"{metric_name:.<30} {value:.4f}")

Parameter 'function'=<function generate_test_responses at 0x78f1959aa3e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/631 [00:00<?, ? examples/s]

First pred IDs: tensor([ 1096,   374,   264, 14633,   315,   264,  8968, 13768,  7055,  5619,
          264, 61584,    13, 19708, 86720,   525,  5619,   279,  1887, 61584])

=== SAMPLE PREDICTIONS ===
PRED:  This is a recording of a bagpipe band playing a melody. Bagpipes are playing the main melody while a drumline is providing the background with percussion. The snare drum, bass drum and cymbals are playing in the background. This piece could be played as the background music at a parade. It could also be played in the soundtrack of a movie where there is a scene taking place in Pakistan. This piece is an amateur recording. There are no other instruments in this song. There are no voices in this song. The tempo of this song is slow. The atmosphere of this song is vibrant and lively.
REF : This is the recording of a Pakistani marching band. The bagpipes are playing the melody. The rhythmic background is provided by a drumline that is played by the snare drums, the bass drums and the c

In [ ]:
del val_set_metrics
del eval_results

## Evaluate on Test Set

In [ ]:
eval_set = test_ds.map(generate_test_responses, batched=True, batch_size=80,
                             remove_columns=test_ds.column_names) # removes unneeded columns
eval_set.set_format(type="torch", columns=["predictions", "references"])
eval_results = compute_metrics(eval_set)

for key, value in eval_results.items():
        metric_name = key.title()
        print(f"{metric_name:.<30} {value:.4f}")

Map:   0%|          | 0/631 [00:00<?, ? examples/s]

First pred IDs: tensor([ 1096,  5492,   374,   264, 73214,    13,  1084,  8471,  1007,   448,
          264,  8593,  7743, 25083,   825,  3409,   323,  1221,   279, 61584])

=== SAMPLE PREDICTIONS ===
PRED:  This song is a salsa. It starts off with a male voice singing one word and then the melody is sung by the male voice. The main melody is played by the keyboard and the trumpet. The bass plays a syncopated rhythm on the first two beats of each measure. The percussion includes a cowbell and a latin style conga. The trumpets start playing a loud salsa melody at the end of the first part of the song. The song is in 4/4 time signature. It can be played at a dance party. The song is in the key of E major. A decrescendo starts from 5
REF : This salsa song features a male voice. It starts off with the keyboard and bass playing a synchronized melody. The percussion is played only as a rimshot in a grouping of two strokes followed by a group of three strokes. This is followed by a male voice

## Save the altered model

In [ ]:
save_path = "/content/gdrive/MyDrive/MusicDescriptorModel_Qwen_3E"
trainer.save_model(save_path)

In [ ]:
from google.colab import runtime
runtime.unassign()